In [5]:
import os
import time
import requests

BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN")
CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID")
GPIO_PIN = 17  # BCM pin number
ALERT_COOLDOWN_SECONDS = 10
SIMULATE_PIR = os.environ.get("PIR_SIMULATION", "1").strip().lower() not in {"0", "false", "no", "off"}
MAX_MOTION_EVENTS = int(os.environ.get("PIR_MAX_MOTION_EVENTS", "1" if SIMULATE_PIR else "0"))

if MAX_MOTION_EVENTS < 0:
    raise ValueError("PIR_MAX_MOTION_EVENTS must be zero or a positive integer.")
if not SIMULATE_PIR and (not BOT_TOKEN or not CHAT_ID):
    raise RuntimeError("Set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in the environment before running this cell.")

TELEGRAM_API = f"https://api.telegram.org/bot{BOT_TOKEN}" if BOT_TOKEN else None
print("PIR simulation mode enabled; Telegram messages will not be sent." if SIMULATE_PIR else "GPIO mode enabled; Telegram credentials loaded.")

PIR simulation mode enabled; Telegram messages will not be sent.


In [6]:
def send_telegram_message(text):
    if SIMULATE_PIR:
        print(f"[dry run] Would send Telegram alert: {text}")
        return None
    response = requests.post(
        f"{TELEGRAM_API}/sendMessage",
        data={"chat_id": CHAT_ID, "text": text},
        timeout=15,
    )
    response.raise_for_status()
    try:
        result = response.json()
    except ValueError as exc:
        raise RuntimeError("Telegram API returned invalid JSON.") from exc
    if not isinstance(result, dict) or result.get("ok") is not True:
        raise RuntimeError(f"Telegram API error: {result}")
    return result["result"]

# Optional: uncomment to verify the bot and chat configuration.
# send_telegram_message("PIR monitor setup check: Telegram alerts are working.")

In [7]:
from datetime import datetime, timezone

last_alert_at = 0.0

def handle_motion():
    global last_alert_at
    now = time.monotonic()
    if now - last_alert_at < ALERT_COOLDOWN_SECONDS:
        return
    timestamp = datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")
    message = f"Motion detected by PIR sensor at {timestamp}."
    try:
        send_telegram_message(message)
        last_alert_at = now
        print(message)
    except requests.RequestException as exc:
        print(f"Telegram request failed: {exc}")
    except RuntimeError as exc:
        print(f"Could not send Telegram alert: {exc}")

if SIMULATE_PIR:
    print("Simulating one PIR motion event.")
    handle_motion()
else:
    from gpiozero import MotionSensor

    pir = None
    event_count = 0
    try:
        pir = MotionSensor(GPIO_PIN, queue_len=1)
        print(f"Monitoring PIR sensor on BCM GPIO {GPIO_PIN}. Interrupt this cell to stop.")
        while MAX_MOTION_EVENTS == 0 or event_count < MAX_MOTION_EVENTS:
            pir.wait_for_motion()
            event_count += 1
            handle_motion()
            pir.wait_for_no_motion()
    except KeyboardInterrupt:
        print("Monitoring stopped.")
    finally:
        if pir is not None:
            pir.close()

Simulating one PIR motion event.
[dry run] Would send Telegram alert: Motion detected by PIR sensor at 2026-11-17T23:43:57+05:30.
Motion detected by PIR sensor at 2026-11-17T23:43:57+05:30.
